# 000_nb_configurations
Shared configuration, reused by every other notebook through `run_notebook()`.

Local counterpart of the Databricks `000_nb_configurations`:

| Databricks | This pipeline |
|---|---|
| Environment from the workspace ID | Environment from `WAHAT_ENV` (default `dev`) |
| Unity Catalog catalogs `bronze_dev`, `silver_dev`, `gold_dev`, `metadata_dev` | Folders `lakehouse/dev/bronze`, `silver`, `gold`, `metadata` |
| `bronze_dev.wahat.gl_lines` | Spark database `bronze_wahat`, table `gl_lines` |
| One secret scope per source connection | One folder per source connection (`RAW_FOLDER`, `MAPPING_FOLDER`) |

Every path can be overridden with an environment variable, so nothing personal is hard-coded.

In [ ]:
import os
import sys
import pathlib
from datetime import datetime

## Environment and folders

In [ ]:
ENV = os.environ.get("WAHAT_ENV", "dev").lower()

PROJECT_ROOT = PIPELINE_ROOT.parent

# Where the raw DD-RET-005 parquet files are (the GL file and the Supporting_Tables folder)
RAW_FOLDER = pathlib.Path(os.environ.get("WAHAT_RAW_FOLDER", PROJECT_ROOT.parent))
# The mapping workbook CSVs
MAPPING_FOLDER = pathlib.Path(os.environ.get("WAHAT_MAPPING_FOLDER", PROJECT_ROOT / "mapping"))
# The lakehouse: every Bronze, Silver, Gold and control table lives under here
LAKEHOUSE = pathlib.Path(os.environ.get("WAHAT_LAKEHOUSE", PROJECT_ROOT / "lakehouse")) / ENV
# Where the publish step writes the parquet files the Power BI model reads: inside the
# lakehouse, per environment, so dev and prod report data never overwrite each other
PUBLISH_FOLDER = pathlib.Path(os.environ.get("WAHAT_PUBLISH_FOLDER", LAKEHOUSE / "publish"))

# Table format: Delta Lake, as on Databricks. "parquet" is only for testing without Delta.
TABLE_FORMAT = os.environ.get("WAHAT_TABLE_FORMAT", "delta").lower()

# source_connection_name in ctrl.data_flow_control -> folder (Databricks: secret scope)
CONNECTIONS = {
    "RAW_FOLDER": RAW_FOLDER,
    "MAPPING_FOLDER": MAPPING_FOLDER,
}

## Windows prerequisites
Spark needs `winutils.exe` and `hadoop.dll` to read and write files on Windows. Put both in
`C:\hadoop\bin` (or set `HADOOP_HOME` to the folder that holds `bin`). See the README.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

if os.name == "nt":
    hadoop_home = pathlib.Path(os.environ.get("HADOOP_HOME", r"C:\hadoop"))
    if not (hadoop_home / "bin" / "winutils.exe").exists():
        raise FileNotFoundError(
            f"winutils.exe not found in {hadoop_home / 'bin'}. "
            "Download winutils.exe and hadoop.dll for Hadoop 3.3 into C:\\hadoop\\bin "
            "(see pipeline/README.md), or set HADOOP_HOME."
        )
    os.environ["HADOOP_HOME"] = str(hadoop_home)
    os.environ["PATH"] = str(hadoop_home / "bin") + os.pathsep + os.environ["PATH"]

## Spark session
One local Spark session per kernel, with Delta Lake switched on. The first run downloads the
Delta jars from Maven Central, so it needs internet once.

In [ ]:
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName(f"wahat_medallion_{ENV}")
    .master("local[*]")
    .config("spark.driver.memory", os.environ.get("WAHAT_DRIVER_MEMORY", "6g"))
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", (LAKEHOUSE / "_warehouse").as_posix())
    .config("spark.ui.showConsoleProgress", "false")
)

if TABLE_FORMAT == "delta":
    from delta import configure_spark_with_delta_pip

    builder = (
        builder
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        # Change Data Feed on every new table: each write also records which rows were
        # inserted, updated or deleted, for any consumer that wants only the changes
        .config("spark.databricks.delta.properties.defaults.enableChangeDataFeed", "true")
    )
    builder = configure_spark_with_delta_pip(builder)

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

## Layer and table naming
Business schema `wahat` sits inside each layer: database `bronze_wahat`, folder
`lakehouse/dev/bronze/wahat/<table>`. Control tables live in database `ctrl`, folder
`lakehouse/dev/metadata/ctrl/<table>`.

In [ ]:
LAYERS = ["bronze", "silver", "gold"]
CTRL_DATABASE = "ctrl"
CTRL_FOLDER = LAKEHOUSE / "metadata" / "ctrl"


def layer_database(layer, business_schema):
    return f"{layer.lower()}_{business_schema}"


def table_full_name(layer, business_schema, table_name):
    return f"{layer_database(layer, business_schema)}.{table_name}"


def table_path(layer, business_schema, table_name):
    return LAKEHOUSE / layer.lower() / business_schema / table_name


def ctrl_table_path(table_name):
    return CTRL_FOLDER / table_name

## Re-register existing tables
A local Spark session has no Unity Catalog, so it forgets table names when the kernel stops.
The data stays in the lakehouse folders; this step points the names back at it.

In [ ]:
def is_table_folder(folder):
    if TABLE_FORMAT == "delta":
        return (folder / "_delta_log").is_dir()
    return any(folder.glob("*.parquet"))


def register_table(database, table_name, folder):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {database}")
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {database}.{table_name} "
        f"USING {TABLE_FORMAT} LOCATION '{folder.as_posix()}'"
    )


def register_existing_tables():
    registered = 0
    for layer in LAYERS:
        for schema_folder in sorted((LAKEHOUSE / layer).glob("*")):
            for table_folder in sorted(schema_folder.glob("*")):
                if table_folder.name.startswith("_"):
                    continue  # staging folder left by an interrupted write
                if table_folder.is_dir() and is_table_folder(table_folder):
                    register_table(layer_database(layer, schema_folder.name), table_folder.name, table_folder)
                    registered += 1
    for table_folder in sorted(CTRL_FOLDER.glob("*")):
        if table_folder.is_dir() and is_table_folder(table_folder):
            register_table(CTRL_DATABASE, table_folder.name, table_folder)
            registered += 1
    return registered


tables_registered = register_existing_tables()

## Summary

In [ ]:
print(f"Environment     : {ENV}")
print(f"Table format    : {TABLE_FORMAT}")
print(f"Spark           : {spark.version}")
print(f"Raw folder      : {RAW_FOLDER}")
print(f"Mapping folder  : {MAPPING_FOLDER}")
print(f"Lakehouse       : {LAKEHOUSE}")
print(f"Publish folder  : {PUBLISH_FOLDER}")
print(f"Tables found    : {tables_registered}")